# Task 1 — Hyperparameter Search Visualisation

Plots training and validation loss curves for all **192 configurations** from the
grid search. Curves are coloured by optimiser family (SGD / Adam) at low opacity
to convey the overall spread; the **best configuration** is highlighted in gold.

> Results are loaded from `.data/Task1_HyperparameterTuning.json`

In [ ]:
import json, os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines  as mlines
from matplotlib.ticker import MaxNLocator

# ── Load results ──────────────────────────────────────────────────────────────
DATA_FILE = os.path.join('.data', 'Task1_HyperparameterTuning.json')
with open(DATA_FILE) as f:
    results = json.load(f)

# ── Identify best config ──────────────────────────────────────────────────────
best      = max(results, key=lambda r: r['best_valid_acc'])
BEST_NAME = best['config']['name']

print(f"Loaded {len(results)} configs")
print(f"Best  : {BEST_NAME}")
print(f"Val Acc: {best['best_valid_acc']*100:.2f}%  |  "
      f"Epochs run: {best['num_epochs']}  |  "
      f"Early stop: {best['stopped_early']}")

In [ ]:
# ── Colour palette ───────────────────────────────────────────────────────────
C_SGD    = '#2166ac'   # deeper blue  — SGD configs
C_ADAM   = '#d94801'   # burnt orange — Adam configs
C_BEST   = '#e63946'   # vivid red    — best configuration
C_BG     = '#ffffff'   # white background
C_GRID   = '#e0e0e0'
C_TEXT   = '#1a1a1a'
C_MUTED  = '#555555'

ALPHA_BACKGROUND = 0.22   # non-best curves
LW_BACKGROUND    = 0.85
LW_BEST          = 2.8

plt.rcParams.update({
    'figure.facecolor' : C_BG,
    'axes.facecolor'   : C_BG,
    'axes.edgecolor'   : C_GRID,
    'axes.labelcolor'  : C_TEXT,
    'axes.titlecolor'  : C_TEXT,
    'axes.grid'        : True,
    'grid.color'       : C_GRID,
    'grid.linewidth'   : 0.5,
    'xtick.color'      : C_MUTED,
    'ytick.color'      : C_MUTED,
    'text.color'       : C_TEXT,
    'legend.facecolor' : '#f5f5f5',
    'legend.edgecolor' : C_GRID,
    'font.family'      : 'DejaVu Sans',
    'font.size'        : 10,
})

print("Plot settings applied.")

In [ ]:
def plot_loss_panel(ax, loss_key, ylabel, title):
    """
    Draw all config curves (background) then the best curve (foreground).

    loss_key : 'train_losses' or 'val_losses'
    """
    # ── Background curves ─────────────────────────────────────────────────────
    for r in results:
        if r['config']['name'] == BEST_NAME:
            continue                          # drawn separately last
        losses = r[loss_key]
        epochs = range(1, len(losses) + 1)
        colour = C_SGD if r['config']['opt_type'] == 'SGD' else C_ADAM
        ax.plot(epochs, losses,
                color=colour, alpha=ALPHA_BACKGROUND,
                linewidth=LW_BACKGROUND, rasterized=True)

    # ── Best config curve ─────────────────────────────────────────────────────
    best_losses = best[loss_key]
    best_epochs = list(range(1, len(best_losses) + 1))
    ax.plot(best_epochs, best_losses,
            color=C_BEST, linewidth=LW_BEST,
            zorder=10, label='_nolegend_')

    # Endpoint marker
    ax.scatter(best_epochs[-1], best_losses[-1],
               color=C_BEST, s=60, zorder=11)

    # ── Annotation box ────────────────────────────────────────────────────────
    cfg  = best['config']
    note = (f"Best config\n"
            f"Opt: {cfg['opt_type']}  lr={cfg['lr']}\n"
            f"Hidden={cfg['hidden_dim']}  drop={cfg['dropout']}\n"
            f"wd={cfg['wd']}\n"
            f"Val acc: {best['best_valid_acc']*100:.2f}%")

    # Position the box to the right or left of the endpoint
    x_end   = best_epochs[-1]
    y_end   = best_losses[-1]
    x_box   = x_end - 5 if x_end > 12 else x_end + 1
    y_box   = y_end + (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.05

    ax.annotate(
        note,
        xy=(x_end, y_end),
        xytext=(x_box, y_box),
        textcoords='data',
        fontsize=7.5,
        color='#1a1a1a',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='#fff8f8',
                  edgecolor=C_BEST, alpha=0.9, linewidth=1.2),
        arrowprops=dict(arrowstyle='->', color=C_BEST, lw=1.2),
        zorder=12,
    )

    # ── Axes formatting ───────────────────────────────────────────────────────
    ax.set_xlabel('Epoch', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=10)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

    # ── Legend ────────────────────────────────────────────────────────────────
    patches = [
        mlines.Line2D([], [], color=C_SGD,  alpha=0.6, lw=1.5, label=f'SGD configs (n={sum(1 for r in results if r["config"]["opt_type"]=="SGD")})'),
        mlines.Line2D([], [], color=C_ADAM, alpha=0.6, lw=1.5, label=f'Adam configs (n={sum(1 for r in results if r["config"]["opt_type"]=="Adam")})'),
        mlines.Line2D([], [], color=C_BEST, lw=LW_BEST,        label='Best configuration'),
    ]
    ax.legend(handles=patches, loc='upper right', fontsize=8, framealpha=0.9)


print("Helper function defined.")

## Figure 1 — Training Loss

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

plot_loss_panel(
    ax,
    loss_key = 'train_losses',
    ylabel   = 'Training Loss (Cross-Entropy)',
    title    = 'Training Loss Across All 192 Hyperparameter Configurations'
)

# Subtitle
fig.text(0.5, 0.01,
    'Each curve = one configuration. Curves coloured by optimiser; '
    'curves that end early were stopped by early stopping (patience = 5).',
    ha='center', fontsize=8, color=C_MUTED, style='italic')

plt.tight_layout(rect=[0, 0.04, 1, 1])

OUT_PATH = os.path.join('.data', 'task1_train_loss.png')
plt.savefig(OUT_PATH, dpi=180, bbox_inches='tight', facecolor='#ffffff')
plt.show()
print(f"Saved → {OUT_PATH}")

## Figure 2 — Validation Loss

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))

plot_loss_panel(
    ax,
    loss_key = 'val_losses',
    ylabel   = 'Validation Loss (Cross-Entropy)',
    title    = 'Validation Loss Across All 192 Hyperparameter Configurations'
)

fig.text(0.5, 0.01,
    'Each curve = one configuration. The best model (gold) is selected '
    'by lowest validation loss over the full search.',
    ha='center', fontsize=8, color=C_MUTED, style='italic')

plt.tight_layout(rect=[0, 0.04, 1, 1])

OUT_PATH = os.path.join('.data', 'task1_val_loss.png')
plt.savefig(OUT_PATH, dpi=180, bbox_inches='tight', facecolor='#ffffff')
plt.show()
print(f"Saved → {OUT_PATH}")

## Bonus — Top 10 Configurations (Zoom)

A cleaner view showing only the top-10 configs by validation accuracy,
with each curve individually labelled. Useful for the report.

In [ ]:
top10 = sorted(results, key=lambda r: r['best_valid_acc'], reverse=True)[:10]

PALETTE = [
    '#e6c229','#50e3c2','#6c8cff','#ff7eb3','#a8e063',
    '#f28e2b','#c084fc','#38bdf8','#fb923c','#34d399'
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=False)

for ax, loss_key, ylabel, title in [
    (axes[0], 'train_losses', 'Training Loss',   'Top-10 Configs — Training Loss'),
    (axes[1], 'val_losses',   'Validation Loss', 'Top-10 Configs — Validation Loss'),
]:
    for rank, (r, colour) in enumerate(zip(top10, PALETTE)):
        losses = r[loss_key]
        epochs = range(1, len(losses) + 1)
        cfg    = r['config']
        lw     = 2.4 if rank == 0 else 1.4
        alpha  = 1.0 if rank == 0 else 0.75

        label = (f"#{rank+1}  {cfg['opt_type']} lr={cfg['lr']} "
                 f"hid={cfg['hidden_dim']} drop={cfg['dropout']} "
                 f"| {r['best_valid_acc']*100:.1f}%")

        ax.plot(epochs, losses, color=colour, lw=lw, alpha=alpha, label=label)

    ax.set_xlabel('Epoch', fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold', pad=10)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))

# Single legend below both panels
handles, labels = axes[1].get_legend_handles_labels()
fig.legend(handles, labels,
           loc='lower center', ncol=2, fontsize=7.5,
           bbox_to_anchor=(0.5, -0.28),
           framealpha=0.9, facecolor='#f5f5f5', edgecolor=C_GRID)

plt.tight_layout()

OUT_PATH = os.path.join('.data', 'task1_top10.png')
plt.savefig(OUT_PATH, dpi=180, bbox_inches='tight', facecolor=C_BG)
plt.show()
print(f"Saved → {OUT_PATH}")

## LaTeX snippet

Copy into your report to include the saved figures:

```latex
\begin{figure}[h]
    \centering
    \includegraphics[width=\linewidth]{.data/task1_train_loss.png}
    \caption{Training loss curves for all 192 hyperparameter configurations.
              SGD configs are shown in blue, Adam in orange.
              The gold curve highlights the best configuration
              (Adam, lr=$10^{-3}$, hidden=100, dropout=0.2, wd=$10^{-4}$).}
    \label{fig:task1_train_loss}
\end{figure}

\begin{figure}[h]
    \centering
    \includegraphics[width=\linewidth]{.data/task1_val_loss.png}
    \caption{Validation loss curves for all 192 hyperparameter configurations.
              The best model is selected by the lowest achieved validation loss.}
    \label{fig:task1_val_loss}
\end{figure}
```